# 88. 数据预处理与Pipeline

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 3 / 34 步：建立训练、切分与预处理工作流**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** Scikit-learn工作流与数据切分  →  **本章任务：** 数据预处理与Pipeline  →  **下一步：** 线性回归与正则化
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

拿到一份真实的表格数据，常常是年龄有空缺、性别和登船口是文字类别，而模型只认识数字。



## 本章目标

学完本章，你将能够：

- **理解**：理解「数据预处理与Pipeline」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「数据预处理与Pipeline」的关键输出指标。
- **迁移**：能把「数据预处理与Pipeline」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 88.1 核心概念

**背景引入**：拿到一份真实的表格数据，常常是年龄有空缺、性别和登船口是文字类别，而模型只认识数字。这一章要解决的就是这类问题——把缺失值补起来、把文字类别转成能参与计算的编码，再把它们与建模步骤串成一条流水线。这样做既避免了把“已经知道答案”的信息偷放进训练，也让同一套预处理能完整地用到新数据上。

- 数值缩放对距离和间隔模型很重要
- OneHotEncoder 将无序类别转成指示变量
- handle_unknown 避免测试集新类别报错
- Pipeline 保证交叉验证时每折独立拟合预处理（打个比方：把清洗、编码、建模封成一条固定工序的“流水线”，每一折都用自己那份原料独立走完，绝不提前“尝”到后面那锅。）


## 88.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 构造混合类型数据 | `pd.read_csv()`、`X.isna()`、`.sum()`、`.to_dict()` | 使用 Titanic 公开数据展示真实缺失值和类别字段。 | 切分前用全量数据计算均值或标准差 |
| 列级预处理流水线 | `pipe.fit()`、`pipe.score()`、`.get_feature_names_out()`、`named_steps['prep']` | 所有填补和编码都封装在流水线中，只在训练集拟合。 | 对名义类别直接使用 1、2、3 表示大小 |


## 88.3 示例 1：构造混合类型数据

使用 Titanic 公开数据展示真实缺失值和类别字段。


<!-- math-foundation:chapter-88 -->
### 数学推导｜标准化与 Pipeline

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜只在训练集估计参数。** 对特征 $j$，计算 $\mu_j^{train}$ 与 $\sigma_j^{train}$。

**第 2 步｜训练、验证和测试共用同一变换。** 

$$
z_{ij}^{split}=\frac{x_{ij}^{split}-\mu_j^{train}}{\sigma_j^{train}}
$$

**第 3 步｜解释泄漏。** 若把验证数据也用于均值计算，实际使用的是

$$
\mu_j^{all}=\frac{n_{train}\mu_j^{train}+n_{valid}\mu_j^{valid}}{n_{train}+n_{valid}}
$$

其中已经含有验证分布信息；Pipeline 的作用就是把“每折只拟合训练部分”固化下来。

**把上面的关系收束为本章计算式：**

$$
z_{ij}=\frac{x_{ij}-\mu_j^{train}}{\sigma_j^{train}}
$$

**符号解释：** 均值和标准差只能从训练数据估计，再应用到验证/测试数据。

**代码对应：** 把 `StandardScaler` 和模型放进同一个 `Pipeline`。

**使用边界：** 切分前标准化会泄漏验证和测试分布。


In [ ]:
import pandas as pd

url = "/datasets/titanic.csv"
df = pd.read_csv(url)
features = ["pclass", "sex", "age", "fare", "embarked"]
X, y = df[features], df["survived"]
print(X.dtypes)
print("缺失值:", X.isna().sum().to_dict())


**练一练**：在示例的基础上只改一处——把字段 `sibsp`（同船兄弟姐妹/配偶数）加进 `features` 列表，再用同样的 `df[features]` 重新取出训练特征，观察缺失值统计的变化。

- 预期：`X` 会多出一列 `sibsp`，且这一列的缺失值通常为 0。
- 改动后重新运行一次 `X.isna().sum()`，对照实际输出与你的预期。


In [ ]:
# 请在下方填写代码
# 任务：把 "sibsp" 加进 features 列表，再用 df[features] 重新取出 X 并统计缺失值。

# TODO: 在下方补全（可参考示例的写法）
#   features.append("sibsp")
#   X = df[features]


In [ ]:
# 完整答案：给 features 增加 sibsp 并重新取出 X
features.append("sibsp")
X = df[features]
print("features =", features)


## 88.4 示例 2：列级预处理流水线

所有填补和编码都封装在流水线中，只在训练集拟合。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

num = ["age", "fare"]
cat = ["pclass", "sex", "embarked"]
prep = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            num,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat,
        ),
    ]
)
pipe = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=500))])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=77
)
pipe.fit(X_train, y_train)
print("测试准确率:", round(pipe.score(X_test, y_test), 3))
try:
    _n_features = len(pipe.named_steps["prep"].get_feature_names_out())
except (AttributeError, TypeError):
    _n_features = sum(
        len(t)
        for _, t, cols in pipe.named_steps["prep"].transformers_
        if t != "drop"
    )
    if _n_features == 0:
        _n_features = pipe.named_steps["prep"]._n_features
print("转换后特征数:", _n_features)


## 88.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 88.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 88.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

reg_X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
reg_y = np.array([12, 15, 19, 23, 27, 31])
reg_baseline = DummyRegressor(strategy="mean").fit(reg_X, reg_y)
reg_model = LinearRegression().fit(reg_X, reg_y)
print("基线预测：", np.round(reg_baseline.predict(reg_X[:2]), 2))
print("模型预测：", np.round(reg_model.predict(reg_X[:2]), 2))
print(
    "基线MAE：", round(mean_absolute_error(reg_y, reg_baseline.predict(reg_X)), 2)
)
print("模型MAE：", round(mean_absolute_error(reg_y, reg_model.predict(reg_X)), 2))


### 88.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
reg_X_changed = reg_X.copy()
reg_X_changed["visits"] = reg_X_changed["visits"] + 1
reg_changed_prediction = reg_model.predict(reg_X_changed)
print("原始前2个预测：", np.round(reg_model.predict(reg_X[:2]), 2))
print("访问次数+1后的预测：", np.round(reg_changed_prediction[:2], 2))
print(
    "预测变化：",
    np.round(reg_changed_prediction[:2] - reg_model.predict(reg_X[:2]), 2),
)


### 88.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 88.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

leak_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
leak_features = [
    column for column in leak_data.columns if column not in forbidden
]
print("禁止使用：", sorted(forbidden))
print("安全特征：", leak_features)
print("原因：特征必须在预测时点已经可获得。")


### 88.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 88.9 易错点提醒

- 切分前用全量数据计算均值或标准差
- 对名义类别直接使用 1、2、3 表示大小
- 测试集出现新类别时编码器报错
- 在训练和预测阶段手工维护两套预处理代码


## 88.10 练习与作业

1. 增加 sibsp 和 parch 两个数值特征
2. 重新拟合流水线
3. 输出新特征数和测试准确率

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 88.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“增加 sibsp 和 parch 两个数值特征”。
2. **独立完成**：不复制示例代码，完成“重新拟合流水线”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出新特征数和测试准确率”，用一两句话说明你修改了什么。

### 88.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 88.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
base_features = ["pclass", "sex", "age", "fare", "embarked"]
features2 = base_features + ["sibsp", "parch"]
num2 = ["age", "fare", "sibsp", "parch"]
prep2 = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            num2,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat,
        ),
    ]
)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    df[features2], y, stratify=y, random_state=77
)
pipe2 = Pipeline(
    [("prep", prep2), ("model", LogisticRegression(max_iter=500))]
).fit(X2_train, y2_train)
practice_score = pipe2.score(X2_test, y2_test)
print("准确率:", round(practice_score, 3))


## 88.12 小结

使用 ColumnTransformer 和 Pipeline 对数值、类别与缺失值进行一致预处理，避免训练测试之间的数据泄漏。


### 88.12.1 你已经掌握

- 识别数值和类别特征
- 分别配置缺失填补、缩放和独热编码
- 用 ColumnTransformer 合并预处理
- 把预处理与模型封装为 Pipeline


### 88.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 88.12.3 需要注意

- 切分前用全量数据计算均值或标准差
- 对名义类别直接使用 1、2、3 表示大小
- 测试集出现新类别时编码器报错
- 在训练和预测阶段手工维护两套预处理代码


### 88.12.4 完成检查

- [ ] 能够识别数值和类别特征
- [ ] 能够分别配置缺失填补、缩放和独热编码
- [ ] 能够用 ColumnTransformer 合并预处理
- [ ] 能够把预处理与模型封装为 Pipeline


### 88.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
